In [ ]:
from bs4 import BeautifulSoup as bs
import string
import numpy as np
from scipy.integrate import quad
import pandas as pd
import scipy.integrate  as ing
import scipy.stats as stats
import math
import numpy.random as rnd
import os
import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk import sent_tokenize
#nltk.download('stopwords')
#nltk.download('punkt')
#nltk.download('punkt_tab')
#nltk.download('wordnet')
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
#from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
from sklearn.model_selection import cross_val_score, StratifiedKFold
from scipy.sparse import hstack
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from scipy.sparse import csr_matrix
from lime.lime_text import LimeTextExplainer
import shap
import random

In [ ]:
df = pd.read_csv('/Users/verak/Downloads/archive/Suicide_Detection.csv')

#Check the column names
#print(df.columns)

#Drop the unwanted index column
df = df.drop(columns=['Unnamed: 0'])

def makeTopicBinary(df):
    df['label'] = 0  #Initialize all as 0
    df.loc[df['class'] == 'suicide', 'label'] = 1  #Set to 1 if class is suicidal
    return df
#print(makeTopicBinary(df))

In [ ]:
def removeEmptyTextClass(df):
    '''
    Removes rows with missing post text or class label.
    '''
    df = df[df['text'].notna()]
    df = df[df['class'].notna()]
    df = df.reset_index(drop=True)
    return df


In [ ]:

def countKeywords(df):
    '''
    Removes punctuation, digits, stopwords, and lemmatizes words.
    Returns:
        countingList: cleaned text per article
        wordsList: all lemmatized tokens from all articles
    '''
    stopWords = set(stopwords.words('english'))
    words_to_keep = {"against", "i", "me", "my", "myself", "because", "before", "being", "below", "between", "down"}

    custom_stopwords = stopWords - words_to_keep

    
    lemmatizer = WordNetLemmatizer()
    
    countingList = []  # cleaned string per article
    wordsList = []     # all words from all articles

    for i in range(len(df)):
        doc_str = str(df.iloc[i]['text']) 
        if pd.isna(doc_str) or doc_str.strip() == "":
            continue

        # Remove punctuation
        no_punctuation = doc_str.translate(str.maketrans(string.punctuation, ' ' * len(string.punctuation)))
        # Remove digits
        clean_text = no_punctuation.translate(str.maketrans('', '', string.digits))
        # Tokenize
        tokens = word_tokenize(clean_text)
        # Keep only words (no symbols or numbers)
        words = [word.lower() for word in tokens if word.isalpha()]
        # Remove stopwords
        filtered_words = [w for w in words if w not in custom_stopwords]
        # Lemmatize
        lemmatized_words = [lemmatizer.lemmatize(w) for w in filtered_words]
        
        # Join back into cleaned string
        lemmatized_text = ' '.join(lemmatized_words)
        countingList.append(lemmatized_text)
        wordsList.extend(lemmatized_words)

    return countingList, wordsList


In [ ]:
def computeTfidf(df, column, max_features=5000):
    texts = df[column].astype(str).tolist()
    vectorizer = TfidfVectorizer(stop_words="english", max_features=max_features)
    X_sparse = vectorizer.fit_transform(texts)
    return X_sparse, vectorizer



In [ ]:
def getBestFeatures_sparse(tfidf_matrix, vectorizer, n):
    means = np.array(tfidf_matrix.mean(axis=0)).flatten()
    feature_names = vectorizer.get_feature_names_out()
    top_n_idx = np.argsort(means)[-n:]
    best_features = [feature_names[i] for i in top_n_idx]
    return best_features
    

In [ ]:
def getTfidf(corpus):
    '''
    purpose: create a Term Document Matrix with the Tfidf scores per word
    input:   document containing content of one or more cases  
    output:  dataframe with Tfidf scores per unique word of the document    
    '''
   
    vectorizer = TfidfVectorizer(use_idf=True)  

   
    vTfidf = vectorizer.fit_transform(corpus)
    dfTfidf = pd.DataFrame(vTfidf[0].T.todense(), index=vectorizer.get_feature_names(),
                columns=["Tfidf"])
    dfTfidf = dfTfidf.sort_values('Tfidf',ascending=False)  
    dfTfidf['Uniq_wds'] = dfTfidf.index 
    dfTfidf.reset_index(drop=True, inplace=True)
 
    return vTfidf
    

In [ ]:
def calcRelFreq(corpus):
    '''
    purpose: create a Term Document Matrix with the relative frequencies
    input:   document containing content of one or more cases  
    output:  dataframe with relative frequencies per unique word of the document    
    ''' 
    # initialize
    vectorizer= CountVectorizer()              

    # code starts here
    vFreq = vectorizer.fit_transform(corpus)
    dfFreq = pd.DataFrame(vFreq[0].T.todense(), index=vectorizer.get_feature_names(),
                columns=["Count"])
    dfFreq = dfFreq.sort_values('Count',ascending=False)  
    dfFreq['Uniq_wds'] = dfFreq.index 
    dfFreq.reset_index(drop=True, inplace=True)
    dfFreq['Rel_freq'] = dfFreq['Count'] / dfFreq.Count.sum()
    #print(dfFreq)
    return dfFreq

In [ ]:
#Predictors
def NaiveBayes(X, y, X_test):
    clf = GaussianNB()
    clf.fit(X, y)    
    y_pred_nb = clf.predict(X_test)
    return y_pred_nb

def logisticRegression(X, y, X_test):
    lr_model = LogisticRegression(solver = 'liblinear', max_iter=500, random_state=246)
    lr_model.fit(X, y)
    y_pred_lr = lr_model.predict(X_test)
    return y_pred_lr

def knn(X, y, X_test):
    #Create KNN Classifier
    knn = KNeighborsClassifier(n_neighbors=7)

    #Train the model using the training sets
    knn.fit(X, y)

    #Predict the response for test dataset
    y_pred_knn = knn.predict(X_test)
    return y_pred_knn

def randomForest(X, y, X_test):
    rf = RandomForestClassifier( n_estimators=500, max_depth=None,  max_features='log2',  min_samples_leaf=2, min_samples_split=2,  random_state=123)
    rf.fit(X, y)
    y_pred_rf = rf.predict(X_test)
    return y_pred_rf


In [ ]:
#def evaluate_multinomial_nb(X_train, X_test, y_train, y_test, alpha: float = 1.0, average: str = "binary"):   

#    clf = MultinomialNB(alpha=alpha)
#    clf.fit(X_train, y_train)

 #   y_pred = clf.predict(X_test)
  #  acc = accuracy_score(y_test, y_pred)
#    precision, recall, f1, _ = precision_recall_fscore_support( y_test, y_pred, average=average)

#    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}, clf
#nb_metrics, nb_model = evaluate_multinomial_nb( X_train, X_test, y_train, y_test, alpha=1.0)
#print(nb_metrics)    

In [ ]:
def evaluate_model(y_test, y_pred):
    print("Accuracy:", accuracy_score(y_test, y_pred) * 100)
    print("F1-score:", f1_score(y_test, y_pred) * 100)
    print("Recall:", recall_score(y_test, y_pred) * 100)
    print("Precision:", precision_score(y_test, y_pred, zero_division='warn') * 100)



In [ ]:
def split_dataset(df, label_col='label'):
    
    train_df, t_df = train_test_split(df, test_size=0.3, random_state=123, stratify=df[label_col])
    val_df, test_df = train_test_split(t_df, test_size=0.3, random_state=123, stratify=t_df[label_col])
    
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)


In [ ]:
def newDf(df):
    stringBodyClean = countKeywords(df)[0]
    df = df.copy()
    df['text'] = stringBodyClean  #Replace the 'text' column 
    return df



In [ ]:
clean_df = removeEmptyTextClass(df)
df_Binary = makeTopicBinary(clean_df)
#print(clean_df)  

train_df, val_df, test_df = split_dataset(df_Binary)

new_train_df = newDf(train_df)
new_val_df = newDf(val_df)
new_test_df = newDf(test_df)

train_df_Binary = makeTopicBinary(new_train_df)
val_df_Binary = makeTopicBinary(new_val_df)
test_df_Binary = makeTopicBinary(new_test_df)

tfidf_train, vec = computeTfidf(train_df_Binary, column='text')
tfidf_val = vec.transform(val_df_Binary['text'].astype(str).tolist())
tfidf_test = vec.transform(test_df_Binary['text'].astype(str).tolist())

#Suicidal notes only
df_suicidal = train_df_Binary[train_df_Binary['label'] == 1]
tfidf_suicidal, vec_suicidal = computeTfidf(df_suicidal, column='text')


In [ ]:
most_used_words= getBestFeatures_sparse(tfidf_suicidal, vec_suicidal, 40)
print(most_used_words)

In [ ]:
df_nonsuicidal = train_df_Binary[train_df_Binary['label'] == 0]
tfidf_nonsuicidal, vec_nonsuicidal = computeTfidf(df_nonsuicidal, column='text')


most_used_words_nons= getBestFeatures_sparse(tfidf_nonsuicidal, vec_nonsuicidal, 40)

print(most_used_words_nons)

In [ ]:
y_train = train_df_Binary['label']
y_val = val_df_Binary['label']
y_test = test_df_Binary['label']

best_indices = [vec.vocabulary_[w] for w in most_used_words if w in vec.vocabulary_]

X_train = tfidf_train[:, best_indices]
X_val = tfidf_val[:, best_indices]
X_test = tfidf_test[:, best_indices]

y_pred_nb = NaiveBayes(X_train.toarray(), y_train, X_val.toarray())
y_pred_lr = logisticRegression(X_train, y_train, X_val)
y_pred_knn = knn(X_train, y_train, X_val)
y_pred_rf = randomForest(X_train, y_train, X_val)



In [ ]:
evaluate_model(y_val, y_pred_nb)

In [ ]:
evaluate_model(y_val, y_pred_lr)

In [ ]:
evaluate_model(y_val, y_pred_knn)

In [ ]:
y_pred_rf = randomForest(X_train, y_train, X_val)

evaluate_model(y_val, y_pred_rf)



In [ ]:
accuracy = []
precision = []
recall = []
f1 = []
k_values = list(range(1, 25))

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_val)
    
    accuracy.append(accuracy_score(y_val, y_pred) * 100)
    precision.append(precision_score(y_val, y_pred, zero_division='warn') * 100)
    recall.append(recall_score(y_val, y_pred) * 100)
    f1.append(f1_score(y_val, y_pred) * 100)

# Create the results table
results_df = pd.DataFrame({
    'k': k_values,
    'Accuracy': accuracy,
    'Precision': precision,
    'Recall': recall,
    'F1': f1
})

print(results_df)

In [ ]:

plt.plot(k_values, f1, linestyle='-', label='F1-score')
plt.plot(k_values, accuracy, linestyle='--', label='Accuracy')
plt.plot(k_values, precision, linestyle='-.', label='Precision')
plt.plot(k_values, recall, linestyle=':', label='Recall')

# Add labels and title
plt.xlabel('Number of Neighbors (k)')
plt.ylabel('Score (%)')
plt.title('Model Evaluation Metrics for KNN with §Different k Values')
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.show()




In [ ]:
max_feature_range = [500, 1000, 3000, 5000]
results = []

for mf in max_feature_range:
    X_tfidf, vec = computeTfidf(df, "text", max_features=mf)
    X_train, X_val, y_train, y_val = train_test_split(
        X_tfidf, df["class"], test_size=0.2, stratify=df["class"]
    )
    
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    
    acc = accuracy_score(y_val, preds)
    f1 = f1_score(y_val, preds, pos_label='suicide', average='binary')
    results.append((mf, acc, f1))

for mf, acc, f1 in results:
    print(f"Features: {mf} | Accuracy: {acc:.4f} | F1-score: {f1:.4f}")


In [ ]:
feature_names = vec_suicidal.get_feature_names_out()
word_to_index = {word: idx for idx, word in enumerate(feature_names)}


keywords = ['suicide', 'suicidal', 'kill', 'die', 'goodbye', 'attempt', 'love', 'time']
for word in keywords:
    if word in word_to_index:
        idx = word_to_index[word]
        word_vector = tfidf_suicidal[:, idx]
        co_occurrence = word_vector.T @ tfidf_suicidal
        co_occurrence_array = co_occurrence.toarray().flatten()

        # Get top 15 co-occurring words
        top_indices = co_occurrence_array.argsort()[::-1]
        top_words = [(feature_names[i], co_occurrence_array[i]) 
                     for i in top_indices if feature_names[i] != word][:15]

        print(f"\nTop co-occurring words with '{word}':")
        for w, score in top_words:
            print(f"{w}: {score:.4f}")

In [ ]:
suicidal_df = train_df_Binary[train_df_Binary['label'] == 1].reset_index(drop=True)
nonsuicidal_df = train_df_Binary[train_df_Binary['label'] == 0].reset_index(drop=True)

# Randomly select one of each
idx_s = random.randint(0, len(suicidal_df) - 1)
idx_n = random.randint(0, len(nonsuicidal_df) - 1)

text_suicidal = suicidal_df.iloc[idx_s]['text']
text_nonsuicidal = nonsuicidal_df.iloc[idx_n]['text']

print("🔴 Suicidal Post:\n", text_suicidal)
print("\n🟢 Non-Suicidal Post:\n", text_nonsuicidal)

In [ ]:
# Define LIME explainer

rf_model = RandomForestClassifier(n_estimators=100, random_state=123)
rf_model.fit(tfidf_train, train_df_Binary["label"])

log_model = LogisticRegression(solver='liblinear', max_iter=500, random_state=246)
log_model.fit(tfidf_train, train_df_Binary["label"])


def predict_rf(texts):
    return rf_model.predict_proba(vec.transform(texts))


class_names = ['Not Suicidal', 'Suicidal']
explainer = LimeTextExplainer(class_names=class_names)

# Model prediction functions using your TF-IDF vectorizer
def predict_rf(texts):
    return rf_model.predict_proba(vec.transform(texts))

def predict_logreg(texts):
    return log_model.predict_proba(vec.transform(texts))

# LIME explanation (Random Forest)
exp_rf_s = explainer.explain_instance(text_suicidal, predict_rf, num_features=8)
exp_rf_s.show_in_notebook()

exp_rf_n = explainer.explain_instance(text_nonsuicidal, predict_rf, num_features=8)
exp_rf_n.show_in_notebook()

# Logistic Regression 
exp_log_s = explainer.explain_instance(text_suicidal, predict_logreg, num_features=8)
exp_log_s.show_in_notebook()

exp_log_n = explainer.explain_instance(text_nonsuicidal, predict_logreg, num_features=8)
exp_log_n.show_in_notebook()



In [ ]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

# Define the parameter grid
param_grid = {
    'n_estimators': [100, 300, 500],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}

# Create the model
rf = RandomForestClassifier(random_state=42)

# Create GridSearchCV
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid,
                           cv=3, scoring='f1', n_jobs=-1, verbose=2)

# Fit to the training data
grid_search.fit(X_train, y_train)

# Print best parameters and evaluate on validation set
print("Best Parameters:", grid_search.best_params_)
best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_val)
print(classification_report(y_val, y_pred))
